In [1]:
!pip install -q transformers peft bitsandbytes sentence-transformers faiss-cpu pandas gradio huggingface_hub accelerate matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.8 MB/s eta 0:00:00


In [2]:
import torch, pandas as pd, numpy as np, faiss, contextlib
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download

HF_REPO_ID = "younginAI/mistral-lol-agent"        # 방금 새 어댑터 올린 곳
BASE_MODEL = "unsloth/mistral-7b-v0.3-bnb-4bit"   # 학습과 동일

csv_path = hf_hub_download(repo_id=HF_REPO_ID, filename="master_meta.csv", repo_type="model")
df = pd.read_csv(csv_path, encoding="utf-8-sig")
POS={"TOP":"Top","JUNGLE":"Jungle","MID":"Mid","ADC":"ADC","SUPPORT":"Support"}
df["context"]=df.apply(lambda x:f"Champion: {x['champion']} | Position: {POS.get(x['position'],x['position'])} | "
    f"Win Rate: {x['win_rate']}% | Tier: {x['tier']} | Core Items: {x['core_items']} | "
    f"Hard Counters: {x['hard_counters']}", axis=1)
documents=df["context"].tolist()

embedder=SentenceTransformer("BAAI/bge-m3")
emb=embedder.encode(documents, normalize_embeddings=True, show_progress_bar=True)
index=faiss.IndexFlatIP(emb.shape[1]); index.add(np.array(emb).astype("float32"))
try:
    from sentence_transformers import CrossEncoder
    reranker=CrossEncoder("BAAI/bge-reranker-v2-m3"); print("✅ reranker on")
except Exception as e:
    reranker=None; print("⚠️ reranker off:", e)

tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL)
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                       bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
base_model=AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
model=PeftModel.from_pretrained(base_model, HF_REPO_ID); model.eval()   # ← HF에서 새 어댑터

ALPACA="""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""
def retrieve(q,k=5,pool=10):
    qe=embedder.encode([q],normalize_embeddings=True)
    _,idx=index.search(np.array(qe).astype("float32"),pool)
    cands=[documents[i] for i in idx[0]]
    if reranker:
        sc=reranker.predict([[q,c] for c in cands])
        cands=[c for _,c in sorted(zip(sc,cands),reverse=True)]
    return cands[:k]
def generate(q, use_rag=True, use_lora=True, k=5):
    ctx="\n".join(retrieve(q,k=k)) if use_rag else ""
    inputs=tokenizer(ALPACA.format(q,ctx),return_tensors="pt").to("cuda")
    cm=contextlib.nullcontext() if use_lora else model.disable_adapter()
    with torch.no_grad(), cm:
        out=model.generate(**inputs,max_new_tokens=256,temperature=0.7,do_sample=True)
    return tokenizer.decode(out[0],skip_special_tokens=True).split("### Response:")[-1].strip()
print("✅ ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


master_meta.csv:   0%|          | 0.00/37.9k [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

✅ ready


In [4]:
import gradio as gr
def respond(message, history): return generate(message, use_rag=True, use_lora=True)
gr.ChatInterface(fn=respond, title="LoL Strategic Expert Agent (RAG + RAFT LoRA)",
    description=" Mistral-7B", theme="glass").launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee083e7e27762f6cc5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
